In [10]:
from agents import OpenAIChatCompletionsModel, AsyncOpenAI
from dotenv import load_dotenv
import os

load_dotenv()


gemini_api_key = os.environ.get('GEMINI_API_KEY')


if not gemini_api_key:
    raise ValueError('Gemini API KEY NOT FOUND')


external_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url='https://generativelanguage.googleapis.com/v1beta/openai'
)


model = OpenAIChatCompletionsModel(
    model = 'gemini-2.5-flash',
    openai_client=external_client
)

In [2]:
from agents import Agent

agent = Agent(
    name = "Assistant",
    model = model
)

In [ ]:
from agents import Runner


result = Runner.run_streamed(starting_agent=agent,input="Tell me 3 jokes about programming or devloper")
print('result is',result)
print('----------------------------')
async for event in result.stream_events():
    print(event)

result is RunResultStreaming:
- Current agent: Agent(name="Assistant", ...)
- Current turn: 0
- Max turns: 10
- Is complete: False
- Final output (NoneType):
    None
- 0 new item(s)
- 0 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResultStreaming` for more details)
----------------------------
AgentUpdatedStreamEvent(new_agent=Agent(name='Assistant', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions=None, prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x000001C06F8E9400>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, ex

In [9]:
import asyncio
from agents import Runner
from openai.types.responses import ResponseTextDeltaEvent


result2 = Runner.run_streamed(agent,'Tell me three jokes about programmer')


async for event in result2.stream_events():
    if event.type == 'raw_response_event' and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


Here are three jokes about programmers:

1.  A programmer's wife asks him to go to the store. "Buy a loaf of bread, and if they have eggs, buy a dozen." The programmer returns with 12 loaves of bread. His wife asks, "Why 12 loaves?!" He replies, "They had eggs."

2.  There are 10 types of people in the world: those who understand binary, and those who don't.

3.  An optimist sees the glass half full. A pessimist sees the glass half empty. A programmer sees the glass as twice as big as it needs to be.

#### With ToolCall

In [3]:
from agents import Runner, function_tool, ItemHelpers
import random


@function_tool
def number_of_jokes():
    return random.randint(1,10)



joke_agent = Agent(
    name="Joker Agent",
    instructions="First run `number_of_jokes` tool, than generate that many jokes",
    model=model,
    tools=[number_of_jokes]
)


result = Runner.run_streamed(starting_agent=joke_agent, input='hello')


async for event in result.stream_events():
    print(event)

AgentUpdatedStreamEvent(new_agent=Agent(name='Joker Agent', handoff_description=None, tools=[FunctionTool(name='number_of_jokes', description='', params_json_schema={'properties': {}, 'title': 'number_of_jokes_args', 'type': 'object', 'additionalProperties': False, 'required': []}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000229E3320040>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)], mcp_servers=[], mcp_config={}, instructions='First run `number_of_jokes` tool, than generate that many jokes', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x00000229E31CD400>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_re

In [12]:
from agents import Runner, function_tool, ItemHelpers
import random


@function_tool
def number_of_jokes():
    return random.randint(1,10)



joke_agent = Agent(
    name="Joker Agents",
    instructions="First run `number_of_jokes` tool, than generate that many jokes",
    model=model,
    tools=[number_of_jokes]
)


result = Runner.run_streamed(starting_agent=joke_agent, input='hello')


async for event in result.stream_events():
    if event.type == 'raw_response_event': # where actual response gonna store
        continue
    
    elif event.type == 'agent_updated_stream_event':
        print(f'New agent Update: {event.new_agent.name}')

    elif event.type == 'run_item_stream_event':
        if event.item.type == 'tool_call_item':
            print(f'-- Tool is called')
        elif event.item.type == 'tool_call_output_item':
            print(f'-- Tool output: {event.item.output}')
        elif event.item.type ==  'message_output_item':
            print(f'--Message output: \n {ItemHelpers.text_message_output(event.item)}')
        else:
            pass

print("--- Run Complete --- ")

New agent Update: Joker Agents
-- Tool is called
-- Tool output: 10
--Message output: 
 Here are 10 jokes for you!

1. Why don't scientists trust atoms? Because they make up everything!
2. What do you call a fake noodle? An impasta!
3. Why did the scarecrow win an award? Because he was outstanding in his field!
4. What do you call a can of soda in the desert? A lost cause!
5. Why don't skeletons fight each other? They don't have the guts.
6. What did the grape say when it got stepped on? Nothing, it just let out a little wine.
7. How do you organize a space party? You "planet"!
8. What's orange and sounds like a parrot? A carrot.
9. Why did the computer go to the doctor? Because it had a virus!
10. I told my wife she was drawing her eyebrows too high. She looked surprised.
--- Run Complete --- 
